# SimpleAI — TinyGPT Additive Mechanistic Probe
### Google Colab Interactive Execution Manual

> **Protocol**: Single-model training, mechanistic diagnostics, dynamic INT8 quantization, and bidirectional Google Drive sync.


## Step 1: Mount Google Drive & Configure Workspace


In [ ]:
from google.colab import drive
import os, sys

# 1. Mount Google Drive (optional)
try:
    drive.mount('/content/drive')
    DRIVE_WORKSPACE = '/content/drive/MyDrive/simpleAI_workspace'
    os.makedirs(f'{DRIVE_WORKSPACE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_WORKSPACE}/runs', exist_ok=True)
    print(f'✓ Google Drive 工作目录就绪: {DRIVE_WORKSPACE}')
except Exception as e:
    print(f'Drive 挂载提示: {e}，将使用 Colab 本地临时存储。')
    DRIVE_WORKSPACE = None

## Step 2: Environment Setup & Codebase Cloning


In [ ]:
# 1. Install core dependencies
!pip install -q torch openpyxl huggingface_hub pandas matplotlib

# 2. 从 Hugging Face 克隆或下载代码
%cd /content
if not os.path.exists('/content/additive-rand-transformer'):
    print('正在从 Hugging Face 克隆仓库...')
    !git clone https://huggingface.co/Hana-ame/additive-rand-transformer /content/additive-rand-transformer

%cd /content/additive-rand-transformer
if '/content/additive-rand-transformer' not in sys.path:
    sys.path.insert(0, '/content/additive-rand-transformer')
print('✓ 代码仓库与 Python 路径已就绪')

## Step 3: Download Pretrained Weights from Hugging Face (.pt Checkpoints)


In [ ]:
import os, torch
from huggingface_hub import hf_hub_download

# Retrieve token from Colab Secrets if private repository
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)

REPO_ID = 'Hana-ame/additive-rand-transformer'
CKPT_NAME = 'l4_d128_cot_bias05_final.pt'
os.makedirs('checkpoints', exist_ok=True)
local_path = f'checkpoints/{CKPT_NAME}'

print(f'正在从 Hugging Face 仓库 ({REPO_ID}) 下载权重: {CKPT_NAME} ...')
try:
    downloaded_file = hf_hub_download(
        repo_id=REPO_ID,
        filename=f'checkpoints/{CKPT_NAME}',
        local_dir='.',
        token=hf_token
    )
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f'✓ 权重下载成功: {local_path} ({size_mb:.2f} MB)')
    
    # 自动备份至 Google Drive
    if DRIVE_WORKSPACE:
        !cp {local_path} {DRIVE_WORKSPACE}/checkpoints/{CKPT_NAME}
        print(f'✓ 已同步备份到 Google Drive')
except Exception as e:
    print(f'下载提示: {e}')

## Step 4: Generate Custom Experiment Configuration (`config.json`)


In [ ]:
import json

custom_config = {
    'layers': 4,                  # Model depth L (1-10)
    'd': 128,                     # 嵌入维度 d (32-512)
    'heads': 4,                   # Attention heads
    'steps': 4000,                # Training steps (建议 2000-4000)
    'batch_size': 32,             # Batch size
    'lr': 3e-4,                   # Learning rate
    'wd': 0.1,                    # Weight decay
    'warmup': 200,                # 预热步数
    'datasource': {
        'type': 'cot',            # cot: 竖式草稿纸 | plain: 无中间过程
        'max_digits': 4,          # 最大操作数位数 (1-4位)
        'bias': 0.5,              # 4位高难度进位加权 (0.5 为最佳相变点)
        'max_spaces': 3,          # 运算符两侧空格随机扰动
        'single': True            # 单样本训练（不跨题打包）
    }
}

with open('config.json', 'w', encoding='utf-8') as f:
    json.dump(custom_config, f, indent=2)

print('✓ 已生成 config.json:')
print(json.dumps(custom_config, indent=2))

## Step 5: Launch Training (`train.py --config config.json`)


In [ ]:
# Execute training and monitor real-time progress and CoT accuracy
!python -m additive_rand_transformer.train --config config.json

## Step 6: Mechanistic Diagnostic Probe (H1 Scratchpad Tamper Test)


In [ ]:
# Execute H1 scratchpad tampering sensitivity probe
!python -m additive_rand_transformer.explore_h1 || true

## Step 7: Dynamic INT8 Quantization Benchmark


In [ ]:
# Execute PyTorch Dynamic INT8 quantization benchmark
!python -m additive_rand_transformer.quantize --checkpoint checkpoints/l4_d128_cot_bias05_final.pt

## Step 8: Interactive Inference & Demonstration (REPL)


In [ ]:
from additive_rand_transformer.model import TinyGPT, TinyGPTConfig
from additive_rand_transformer.data import BOS, EOS, SP, PLUS, MINUS, _int_to_tokens, decode

ckpt_file = 'checkpoints/l4_d128_cot_bias05_final.pt'
if not os.path.exists(ckpt_file):
    import glob
    ckpts = sorted(glob.glob('runs/**/checkpoint*.pt', recursive=True))
    if ckpts:
        ckpt_file = ckpts[-1]

if os.path.exists(ckpt_file):
    ck = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    cfg = TinyGPTConfig(**{k: v for k, v in ck['config'].items() if k in TinyGPTConfig.__dataclass_fields__})
    model = TinyGPT(cfg)
    model.load_state_dict(ck['model'])
    model.eval()
    print(f'✓ Model successfully loaded: {ckpt_file} (L={cfg.n_layer}, d={cfg.n_embd}, {model.num_parameters():,} 参数)')

    def calculate(a, b, op='+'):
        op_id = PLUS if op == '+' else MINUS
        prompt = [BOS] + _int_to_tokens(a) + [SP, op_id, SP] + _int_to_tokens(b) + [SP]
        ids = list(prompt)
        with torch.no_grad():
            for _ in range(80):
                x = torch.tensor([ids], dtype=torch.long)
                logits, _ = model(x, None)
                nxt = int(logits[0, -1].argmax())
                ids.append(nxt)
                if nxt == EOS:
                    break
        result_str = decode(ids)
        print(f'【题目】: {a} {op} {b}')
        print(f'【模型生成 (含竖式CoT)】:\n{result_str}\n')

    # 演示多位计算
    calculate(37, 85, '+')
    calculate(523, 194, '-')
    calculate(1234, 5678, '+')
    calculate(9999, 4321, '-')
else:
    print('未找到 checkpoint，请先运行步骤 3 或步骤 5。')

## Step 9: Archive All Artifacts to Google Drive


In [ ]:
if DRIVE_WORKSPACE:
    !cp -ru runs/ {DRIVE_WORKSPACE}/runs/ || true
    !cp -ru checkpoints/ {DRIVE_WORKSPACE}/checkpoints/ || true
    !cp config.json {DRIVE_WORKSPACE}/ || true
    print(f'✓ All weights, configurations, and logs successfully archived to Google Drive: {DRIVE_WORKSPACE}')
else:
    print('未挂载 Google Drive，产物保存在 Colab 本地。')